# 📸 Lab 02 · What a single photo can't tell you

**World Models course · Lectures 2, 5, 6 and 11 · HW1 Part B** &nbsp;|&nbsp; ⏱ about 50 min &nbsp;|&nbsp; 💻 CPU only

Look at a photo of a ball in mid-air. Is it going up or coming down? **You can't know.** The photo hides the ball's velocity.

A world model has the same problem: it needs a **state** (everything that matters for the future), but a camera only gives **observations**. In this lab you will:

1. Build tiny videos where two different futures share the **exact same** middle frame.
2. Test which "features" an encoder must keep in order to predict the future.
3. See why **looking good** (reconstruction) is not the same as **knowing what happens next** (prediction).
4. Use **memory** to track a ball while it is hidden.
5. Discover why **averaging** image features can erase *where* things are. This is why robot world models such as DINO-WM keep spatial patch features.

🧩 challenges · 🔮 predictions · 🎛️ playgrounds. Empty or wrong blanks never break the notebook: it explains, then uses a working answer.

In [ ]:
#@title 🔧 Step 0 · Run this cell first (click ▶). It loads the tools for this lab. { display-mode: "form" }
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.kernel_ridge import KernelRidge
plt.rcParams.update({"figure.dpi": 110})
SIZE = 24
_grid = np.linspace(0, 1, SIZE)
xx, yy = np.meshgrid(_grid, _grid)
def render(p):
    return np.exp(-((xx - p[0]) ** 2 + (yy - p[1]) ** 2) / (2 * 0.05 ** 2))

# ---------------------------------------------------------------------------
# Guided-lab helpers. You never need to edit this cell.
#  * ___            : a blank for you to fill in
#  * check(name, x) : checks your answer; if it is blank or wrong, it explains
#                     and hands back a working version so the notebook keeps going
#  * quiz(id)       : a clickable multiple-choice question
#  * playground(...) : sliders that re-run a function when you let go
# ---------------------------------------------------------------------------
import inspect, html as _html
import numpy as np
from IPython.display import display, HTML
import os
try:
    import ipywidgets as widgets
    _WIDGETS = not os.environ.get("GUIDE_NO_WIDGETS")
except Exception:
    _WIDGETS = False

class BlankNotFilled(Exception):
    pass

class _Blank:
    """The ___ placeholder. Any maths with it stops with a friendly message."""
    __array_ufunc__ = None
    def _stop(self, *args, **kwargs):
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop
    __truediv__ = __rtruediv__ = __floordiv__ = __rfloordiv__ = _stop
    __pow__ = __rpow__ = __matmul__ = __rmatmul__ = __mod__ = __rmod__ = _stop
    __neg__ = __pos__ = __abs__ = __getitem__ = __call__ = __iter__ = _stop
    __lt__ = __le__ = __gt__ = __ge__ = __bool__ = __float__ = __int__ = __index__ = _stop
    __array__ = __len__ = _stop
    def __getattr__(self, name):
        if name.startswith('__'):
            raise AttributeError(name)
        raise BlankNotFilled("There is still a ___ blank to fill in.")
    def __repr__(self):
        return "___"

___ = _Blank()
CHALLENGES, QUIZZES = {}, {}
_solved, _quiz_score = {}, {}

_STYLE = {
    "ok":   ("#e8f6ee", "#1b7a4b", "✅"),
    "wait": ("#fff5e0", "#9a5b00", "🧩"),
    "bad":  ("#fdecea", "#b3261e", "❌"),
    "info": ("#eaf1fb", "#245eb5", "💡"),
}

def card(kind, title, body=""):
    bg, fg, icon = _STYLE[kind]
    display(HTML(
        f'<div style="background:{bg};border-left:5px solid {fg};padding:10px 14px;'
        f'border-radius:6px;margin:6px 0;color:#1d2530;font-size:14px;line-height:1.5">'
        f'<b style="color:{fg}">{icon} {title}</b><div>{body}</div></div>'))

def _as_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    if isinstance(x, (list, tuple)):
        return [_as_numpy(v) for v in x]
    return x

def _same(a, b, tol):
    a, b = _as_numpy(a), _as_numpy(b)
    if isinstance(a, list) or isinstance(b, list):
        return isinstance(a, list) and isinstance(b, list) and len(a) == len(b) and all(_same(x, y, tol) for x, y in zip(a, b))
    try:
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
    except Exception:
        return a == b
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def _has_blank(obj):
    if isinstance(obj, _Blank):
        return True
    if callable(obj):
        try:
            return "___" in inspect.getsource(obj)
        except Exception:
            return False
    return False

def check(name, answer):
    """Check a challenge. Returns your answer if it works, otherwise a working reference."""
    ch = CHALLENGES[name]
    ref = ch["reference"]
    title = ch.get("title", name)
    fallback = ("<br><i>For now the notebook will use a working version so every later cell still runs. "
                "Come back, fill it in, and re-run this cell.</i>")
    if _has_blank(answer):
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” is waiting for you", "Hint: " + ch["hint"] + fallback)
        return ref
    try:
        if "test" in ch:
            ok, message = ch["test"](answer)
        elif callable(ref):
            ok, message = True, ""
            for args in ch["cases"]:
                args = args if isinstance(args, tuple) else (args,)
                expected, got = ref(*args), answer(*args)
                if not _same(expected, got, ch.get("tol", 1e-6)):
                    ok = False
                    message = "For a test input your function gave a different result from the expected one."
                    break
        else:
            ok = _same(ref, answer, ch.get("tol", 1e-6))
            message = f"You entered <code>{_html.escape(repr(_as_numpy(answer)))}</code>."
    except BlankNotFilled:
        _solved.setdefault(name, False)
        card("wait", f"Challenge “{title}” still has a blank", "Hint: " + ch["hint"] + fallback)
        return ref
    except Exception as err:
        ok, message = False, f"Running your version raised <code>{_html.escape(type(err).__name__)}: {_html.escape(str(err))}</code>."
    if ok:
        _solved[name] = True
        card("ok", f"Challenge solved: {title}", ch.get("why", ""))
        return answer
    _solved[name] = False
    card("bad", f"Not quite yet: {title}", message + "<br>Hint: " + ch["hint"] + fallback)
    return ref

def quiz(qid):
    q = QUIZZES[qid]
    question = f'<div style="font-size:15px;margin:8px 0 4px"><b>{"🔮 Predict: " if q.get("predict") else "🤔 "}{q["q"]}</b></div>'
    if not _WIDGETS:
        options = "".join(f"<li>{_html.escape(o)}</li>" for o in q["options"])
        display(HTML(question + f"<ol type='A'>{options}</ol><details><summary>Answer</summary>"
                     f"{'ABCDEFG'[q['answer']]}. {q['explain']}</details>"))
        return
    out = widgets.Output()
    buttons = []
    def choose(i):
        def handler(_):
            _quiz_score.setdefault(qid, i == q["answer"])
            for j, b in enumerate(buttons):
                b.button_style = "success" if j == q["answer"] else ("danger" if j == i else "")
            with out:
                out.clear_output()
                if i == q["answer"]:
                    card("ok", "Yes!", q["explain"])
                else:
                    card("bad", "Not this one. Here is the reasoning:", q["explain"])
        return handler
    for i, option in enumerate(q["options"]):
        b = widgets.Button(description=f"{'ABCDEFG'[i]}. {option}", layout=widgets.Layout(width="auto", max_width="100%"))
        b.on_click(choose(i))
        buttons.append(b)
    display(HTML(question), widgets.VBox(buttons), out)

def playground(fn, **controls):
    """controls: name=(min, max, step, default) for sliders, or name=[option, ...] for a dropdown."""
    defaults, sliders = {}, {}
    for name, spec in controls.items():
        if isinstance(spec, list):
            defaults[name] = spec[0]
            if _WIDGETS:
                sliders[name] = widgets.Dropdown(options=spec, value=spec[0], description=name)
        else:
            lo, hi, step, value = spec
            defaults[name] = value
            if _WIDGETS:
                kind = widgets.IntSlider if all(isinstance(v, int) for v in spec) else widgets.FloatSlider
                sliders[name] = kind(min=lo, max=hi, step=step, value=value, description=name,
                                     continuous_update=False, style={"description_width": "initial"},
                                     layout=widgets.Layout(width="420px"))
    if _WIDGETS:
        ui = widgets.VBox(list(sliders.values()))
        out = widgets.interactive_output(fn, sliders)
        display(ui, out)
    else:
        fn(**defaults)

def progress_report():
    solved = sum(_solved.values()); total = len(CHALLENGES)
    right = sum(_quiz_score.values()); asked = len(_quiz_score)
    stars = "⭐" * solved + "☆" * (total - solved)
    body = f"Challenges solved yourself: <b>{solved} / {total}</b> {stars}<br>"
    body += f"Quiz questions right on the first click: <b>{right} / {asked}</b> (of {len(QUIZZES)} in this lab)"
    missing = [CHALLENGES[k].get('title', k) for k in CHALLENGES if not _solved.get(k)]
    if missing:
        body += "<br>Still worth a try: " + ", ".join(missing)
    card("info", "Your progress in this lab", body)

# ---- this lab's challenges and quizzes ----
def _ref_clip(p, v):
    return np.array([render(p - v), render(p), render(p + v)])
CHALLENGES["clip"] = dict(title="Film three frames", reference=_ref_clip,
    cases=[(np.array([0.5, 0.5]), np.array([0.07, -0.02])), (np.array([0.3, 0.6]), np.array([-0.05, 0.05]))],
    hint="One step <i>before</i> now, the ball was at position − velocity. One step <i>after</i>, it will be at position + velocity.",
    why="Each frame is a snapshot of the same moving ball at a different time.")

def _ref_center(images):
    mass = images.sum(axis=(-2, -1))
    return np.stack([(images * xx).sum(axis=(-2, -1)) / mass, (images * yy).sum(axis=(-2, -1)) / mass], axis=-1)
CHALLENGES["center"] = dict(title="Find the ball", reference=_ref_center,
    cases=[np.array([render(np.array([0.3, 0.7])), render(np.array([0.6, 0.4]))])], tol=1e-6,
    hint="The centre of mass is the brightness-weighted average of x: add up (image × xx), then divide by the total brightness.",
    why="A <b>centre of mass</b> turns a whole picture into two numbers: where the ball is. It is a hand-designed <i>encoder</i>.")

CHALLENGES["motion"] = dict(title="Estimate velocity from two frames", reference=lambda now, before: now - before,
    cases=[(np.array([[0.5, 0.4]]), np.array([[0.45, 0.42]]))],
    hint="Velocity is how far the ball moved between the previous frame and the current frame.",
    why="Two frames reveal motion that one frame hides. This is why video models and robot policies are fed a <b>history</b> of frames.")

CHALLENGES["dead_reckon"] = dict(title="Remember while hidden", reference=lambda last_pos, last_vel, steps_hidden: last_pos + steps_hidden * last_vel,
    cases=[(np.array([0.2, 0.3]), np.array([0.01, -0.02]), 7)],
    hint="If the ball keeps its velocity, after <i>n</i> hidden steps it has moved n × velocity from where you last saw it.",
    why="Carrying a belief forward without new observations is what a recurrent world model's <b>hidden state</b> does (lecture 11: RSSM in Dreamer).")

CHALLENGES["pool"] = dict(title="Global average pooling", reference=lambda patches: patches.mean(axis=(1, 2)),
    cases=[np.arange(2 * 4 * 4 * 3, dtype=float).reshape(2, 4, 4, 3)],
    hint="Average over the two grid axes (rows and columns of patches) but keep the batch axis and feature axis: <code>axis=(1, 2)</code>.",
    why="Pooling makes one summary vector per image. It is great for <i>what</i> is in the image and can be terrible for <i>where</i>.")

QUIZZES["alias"] = dict(predict=True, q="Two clips have the identical current frame but opposite futures. Could any model, however big, predict the future from the current frame alone?",
    options=["Yes, with enough training data", "Yes, if it is a transformer", "No: identical inputs must give identical outputs"],
    answer=2, explain="A model is a function. Same input → same output. When one observation can come from different states, this is <b>state aliasing</b>. The fix is more information (history, other sensors), not a bigger model.")
QUIZZES["best_feature"] = dict(predict=True, q="Which feature set will predict the ball's next position best?",
    options=["Total brightness", "Current position only", "Current position + motion from the previous frame"],
    answer=2, explain="Brightness says nothing about location. Position alone can only guess the <i>average</i> of the two futures. Adding motion removes the ambiguity.")
QUIZZES["recon"] = dict(q="PCA reconstructs the current frame very well. Does that mean its code can predict the future?",
    options=["Yes: a perfect picture contains everything", "No: the current frame itself does not contain the velocity"],
    answer=1, explain="Reconstruction measures how well you <i>copy what you saw</i>. Prediction needs information that may not be in the frame at all. The two can disagree, and this gap is a big theme of lectures 5 and 6 (JEPA).")
QUIZZES["pooling"] = dict(predict=True, q="We summarise each image by <b>averaging</b> all its patch features into one vector. Can a probe still recover where the ball is?",
    options=["Yes, perfectly", "Much worse: averaging throws away the layout", "Only if the ball is large"],
    answer=1, explain="The average of a shuffled grid equals the average of the original grid. Layout is gone. Keeping even a coarse 2×2 or 4×4 grid restores position information.")
print('✅ Setup complete. Scroll down and run the cells in order.')

---
## 1 · Film a tiny video 🎞️

`render(p)` (already loaded) draws a soft white ball at position `p = (x, y)` on a 24×24 image. Positions go from 0 to 1.

A **clip** is three frames: *previous*, *current*, *future*. The ball moves by velocity `v` each step.

### 🧩 Challenge 1 · Film three frames

In [ ]:
def make_clip(p, v):
    previous = render(___)     # 🧩 where was the ball one step ago?
    current  = render(p)
    future   = render(___)     # 🧩 where will it be one step from now?
    return np.array([previous, current, future])

make_clip = check("clip", make_clip)

clip = make_clip(np.array([0.4, 0.5]), np.array([0.1, 0.05]))
fig, axs = plt.subplots(1, 3, figsize=(7, 2.6))
for ax, frame, name in zip(axs, clip, ["previous", "current", "future"]):
    ax.imshow(frame, origin="lower", cmap="magma"); ax.set_title(name); ax.axis("off")
plt.show()

<details><summary>🤔 <b>Need a hint?</b></summary>

Before = position minus velocity. After = position plus velocity.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>previous = render(p - v)     # 🧩 where was the ball one step ago?
future   = render(p + v)     # 🧩 where will it be one step from now?</pre>

</details>

---
## 2 · The trap: same photo, different futures 🪤

We create **600 pairs** of clips. In each pair, the ball sits at the *same* position in the current frame, but moves in **opposite directions**. The current frames are pixel-for-pixel identical.

In [ ]:
rng = np.random.default_rng(7)
positions = rng.uniform(0.25, 0.75, (600, 2))
clips, futures, groups = [], [], []
for g, p in enumerate(positions):
    direction = rng.normal(size=2)
    direction = direction / np.linalg.norm(direction) * 0.075      # a speed of 0.075 per step
    for sign in (-1, +1):                                           # the same position, opposite motion
        v = sign * direction
        clips.append(make_clip(p, v)); futures.append(p + v); groups.append(g)
clips, futures, groups = np.array(clips), np.array(futures), np.array(groups)

print("Current frames identical within a pair?", np.array_equal(clips[0, 1], clips[1, 1]))
print("Futures identical?                      ", np.allclose(futures[0], futures[1]))

# Split by PAIR, so the two twins of a pair are never on different sides of the split.
order = rng.permutation(600)
train = np.isin(groups, order[:420]); test = np.isin(groups, order[500:])

fig, axs = plt.subplots(2, 3, figsize=(7, 5))
for row in range(2):
    for t, name in enumerate(["previous", "current", "future"]):
        axs[row, t].imshow(clips[row, t], origin="lower", cmap="magma"); axs[row, t].axis("off")
        axs[row, t].set_title(f"clip {'AB'[row]} · {name}")
plt.suptitle("Identical middle frames, opposite futures"); plt.tight_layout(); plt.show()

In [ ]:
quiz("alias")

---
## 3 · What should an encoder keep? 🔍

An **encoder** turns an image into a few useful numbers (features). We try three hand-made encoders and ask each to predict where the ball will be next:

| Encoder | Numbers kept |
|---|---|
| brightness | 1: total brightness |
| position | 2: centre of mass (x, y) of the current frame |
| position + motion | 4: position, plus how far it moved since the previous frame |

### 🧩 Challenge 2 · Find the ball
The centre of mass is a **brightness-weighted average** of the pixel coordinates (`xx` holds each pixel's x, `yy` each pixel's y).

In [ ]:
def center_of_mass(images):
    mass = images.sum(axis=(-2, -1))                       # total brightness of each image
    x = ___          # 🧩 brightness-weighted average of xx
    y = (images * yy).sum(axis=(-2, -1)) / mass
    return np.stack([x, y], axis=-1)

center_of_mass = check("center", center_of_mass)
print("Ball drawn at (0.30, 0.70) → found at", np.round(center_of_mass(render(np.array([0.3, 0.7]))), 3))

<details><summary>🤔 <b>Need a hint?</b></summary>

Same as the y line below it, but with <code>xx</code>.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>x = (images * xx).sum(axis=(-2, -1)) / mass          # 🧩 brightness-weighted average of xx</pre>

</details>

### 🧩 Challenge 3 · Estimate velocity from two frames

In [ ]:
def motion(now, before):
    return ___        # 🧩 how far did it move in one step?

motion = check("motion", motion)

now_pos, before_pos = center_of_mass(clips[:, 1]), center_of_mass(clips[:, 0])
encoders = {
    "brightness":        clips[:, 1].sum(axis=(-2, -1))[:, None],
    "position":          now_pos,
    "position + motion": np.c_[now_pos, motion(now_pos, before_pos)],
}

<details><summary>🤔 <b>Need a hint?</b></summary>

Displacement = where it is now minus where it was.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return now - before        # 🧩 how far did it move in one step?</pre>

</details>

In [ ]:
quiz("best_feature")

### The probe test
A **probe** is a small, simple model (here linear regression) trained on top of fixed features. If even a simple probe can read the answer, the features clearly *contain* it.

In [ ]:
errors = {}
for name, feats in encoders.items():
    probe = Ridge(alpha=1e-3).fit(feats[train], futures[train])
    guess = probe.predict(feats[test])
    errors[name] = np.linalg.norm(guess - futures[test], axis=1).mean()
    print(f"{name:>18}: average miss = {errors[name]:.4f}")

plt.figure(figsize=(6, 2.6))
plt.barh(list(errors), list(errors.values()), color=["#bbb", "#f2a65a", "#3b8ea5"])
plt.axvline(0.075, ls="--", c="k"); plt.text(0.076, 0.1, "one step of motion", fontsize=8)
plt.xlabel("average distance between predicted and true future"); plt.title("More information beats more guessing"); plt.show()

**Read the chart:** "position only" misses by about **one step of motion** (0.075). Its best strategy is to guess *"the ball stays where it is"*, halfway between the two possible futures. Motion from history removes the ambiguity.

---
## 4 · Looking good ≠ predicting well 🎨

**PCA** is the simplest compression algorithm: it squeezes each 576-pixel image into `k` numbers and can decode them back. Think of it as a tiny, transparent autoencoder.

PCA numbers relate to position in a *curvy* way, so a straight-line probe would underestimate what they contain. Here we use a **flexible probe** (kernel ridge regression), which can follow curves.

> 💡 **Research habit:** if a simple probe fails, the information may still be there. Try a more flexible probe before concluding *"the features don't know."*

### 🎛️ Playground · How many numbers do you need?
Drag `k`. Watch the reconstruction sharpen, then compare future prediction from **one** frame's code with **two** frames' codes.

In [ ]:
flat_now  = clips[:, 1].reshape(len(clips), -1)       # current frames as 576-number vectors
flat_prev = clips[:, 0].reshape(len(clips), -1)

def flexible_probe(codes):
    probe = KernelRidge(alpha=1e-3, kernel="rbf", gamma=0.5 / codes.shape[1])   # a probe that can bend
    probe.fit(codes[train], futures[train])
    return np.linalg.norm(probe.predict(codes[test]) - futures[test], axis=1).mean()

def compress(k=16):
    pca = PCA(n_components=k, random_state=0).fit(flat_now[train])      # learn compression on TRAIN only
    code = pca.transform(flat_now)                                      # k numbers per image
    recon = pca.inverse_transform(code).reshape(-1, SIZE, SIZE)
    recon_err = np.mean((recon[test] - clips[test, 1]) ** 2)
    both = np.c_[code, pca.transform(flat_prev)]                         # the same encoder on two frames

    i = np.flatnonzero(test)[0]
    fig, axs = plt.subplots(1, 2, figsize=(5, 2.6))
    axs[0].imshow(clips[i, 1], origin="lower", cmap="magma"); axs[0].set_title("original")
    axs[1].imshow(recon[i], origin="lower", cmap="magma"); axs[1].set_title(f"from {k} numbers")
    for a in axs: a.axis("off")
    plt.show()
    print(f"reconstruction error            : {recon_err:.5f}")
    print(f"future miss from 1 frame's code : {flexible_probe(code):.4f}   (one step of motion = 0.075)")
    print(f"future miss from 2 frames' codes: {flexible_probe(both):.4f}")

playground(compress, k=(2, 64, 2, 16))

In [ ]:
quiz("recon")

---
## 5 · Memory: tracking a hidden ball 🙈

Now a ball rolls in a straight line, but an **occluder** hides it for a while. A model with **no memory** can only say *"I don't know"* and guesses the middle of the screen. A model **with memory** remembers the last position and velocity it saw and keeps moving its belief forward.

### 🧩 Challenge 4 · Remember while hidden

In [ ]:
def remembered_position(last_pos, last_vel, steps_hidden):
    return ___      # 🧩 carry the belief forward

remembered_position = check("dead_reckon", remembered_position)

<details><summary>🤔 <b>Need a hint?</b></summary>

Keep moving at the last known velocity for however many steps it has been hidden.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return last_pos + steps_hidden * last_vel      # 🧩 carry the belief forward</pre>

</details>

### 🎛️ Playground · Occlusion and sensor noise
* `hidden_steps`: how long the ball is hidden.
* `sensor_noise`: how shaky our position measurements are. Noise makes the velocity estimate wrong, and memory errors then **grow** with time hidden.

In [ ]:
def track(hidden_steps=10, sensor_noise=0.005, seed=0):
    r = np.random.default_rng(seed)
    p0, v = np.array([0.1, 0.2]), np.array([0.03, 0.02])
    T = 8 + hidden_steps + 4
    truth = np.array([p0 + t * v for t in range(T)])
    seen = truth + r.normal(0, sensor_noise, truth.shape)          # noisy measurements
    visible = np.ones(T, bool); visible[8:8 + hidden_steps] = False

    memory, no_memory = [], []
    last_pos, last_vel, hidden_for = seen[0], np.zeros(2), 0
    for t in range(T):
        if visible[t]:
            if t > 0: last_vel = seen[t] - last_pos if hidden_for == 0 else last_vel
            last_pos, hidden_for = seen[t], 0
            memory.append(seen[t]); no_memory.append(seen[t])
        else:
            hidden_for += 1
            memory.append(remembered_position(last_pos, last_vel, hidden_for))
            no_memory.append(np.array([0.5, 0.5]))                  # no memory: guess the middle
    memory, no_memory = np.array(memory), np.array(no_memory)
    hid = ~visible
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(*truth.T, "k-", lw=3, alpha=0.3, label="true path")
    ax.plot(*truth[hid].T, "ko", ms=4, alpha=0.4, label="true while hidden")
    ax.plot(*memory[hid].T, "o", c="tab:blue", ms=4, label="memory belief")
    ax.plot(*no_memory[hid][:1].T, "x", c="tab:red", ms=10, label="no-memory guess")
    ax.set(xlim=(0, 1), ylim=(0, 1), title="Tracking through an occluder"); ax.legend(fontsize=8); plt.show()
    print(f"average error while hidden · memory: {np.linalg.norm(memory[hid] - truth[hid], axis=1).mean():.3f}"
          f" · no memory: {np.linalg.norm(no_memory[hid] - truth[hid], axis=1).mean():.3f}")

playground(track, hidden_steps=(1, 20, 1, 10), sensor_noise=(0.0, 0.03, 0.0025, 0.005), seed=(0, 20, 1, 0))

---
## 6 · Why averaging can erase *where* 🧱

Modern vision encoders (ViT, DINOv2) cut an image into **patches** and produce one feature vector per patch. You can then either:
* **average** all patches into one vector (*global pooling*), or
* keep a **grid** of patch features (*spatial features*).

We fake a patch encoder: cut each image into a 4×4 grid and describe each patch by 3 numbers.

### 🧩 Challenge 5 · Global average pooling

In [ ]:
def patch_features(images, grid=4):
    n, s = len(images), SIZE // grid
    patches = images.reshape(n, grid, s, grid, s).transpose(0, 1, 3, 2, 4)    # (n, 4, 4, 6, 6)
    return np.stack([patches.mean((-2, -1)), patches.max((-2, -1)), patches.std((-2, -1))], axis=-1)  # (n, 4, 4, 3)

def global_pool(patches):
    return ___     # 🧩 average over the grid rows and columns

global_pool = check("pool", global_pool)

<details><summary>🤔 <b>Need a hint?</b></summary>

The patch grid lives on axes 1 and 2 of the (n, 4, 4, 3) array.

</details>
<details><summary>🔑 <b>Show the answer</b> (try first!)</summary>

<pre>return patches.mean(axis=(1, 2))     # 🧩 average over the grid rows and columns</pre>

</details>

In [ ]:
quiz("pooling")

In [ ]:
feats = patch_features(clips[:, 1])            # current frames only
now = center_of_mass(clips[:, 1])              # the true current position, the thing we want to read out
def two_by_two(p):                              # average into a coarse 2×2 grid, then flatten
    return p.reshape(-1, 2, 2, 2, 2, 3).mean(axis=(2, 4)).reshape(len(p), -1)
options = {
    "global average (3 numbers)": global_pool(feats),
    "2×2 grid (12 numbers)":      two_by_two(feats),
    "full 4×4 grid (48 numbers)": feats.reshape(len(feats), -1),
}
for name, f in options.items():
    probe = Ridge(alpha=1e-3).fit(f[train], now[train])
    miss = np.linalg.norm(probe.predict(f[test]) - now[test], axis=1).mean()
    print(f"{name:>28}: position miss = {miss:.4f}")

**What you should see:** the global average barely knows where the ball is. The spatial grids recover much more.

⚠️ **Honest caveat:** our fake encoder is simple. Real DINOv2 features can keep *some* position information even after pooling. Whether they keep **enough** for a robot is an experiment, not an assumption. That is exactly what the PushT capstone measures.

---
## 7 · Recap and industry links 🏭

| You saw | Name in research | Where it matters |
|---|---|---|
| Identical frames, different futures | state aliasing / partial observability | every robot policy stacks several frames or adds joint sensors (π0, GR00T, Diffusion Policy) |
| Features that keep what's needed | representation learning | Meta **DINOv2 / V-JEPA 2** encoders power world models and robot planners |
| Great reconstruction, poor prediction | pixel vs. latent objectives | why **JEPA** predicts *features* instead of painting pixels |
| Belief carried through occlusion | recurrent hidden state | Dreamer's RSSM, video world models with long memory (Genie 3) |
| Pooling erases location | global vs. spatial tokens | **DINO-WM** keeps patch tokens to plan pushes precisely |

### 🧪 HW1 Part B, beginner version
1. In the PCA playground, try `k=2` and `k=64`. Does better reconstruction ever let the **1-frame** code beat 0.075? Why not?
2. In the occlusion playground, raise `sensor_noise`. Why does memory error grow the longer the ball is hidden?
3. Replace the 4×4 grid with an 8×8 grid in `patch_features(images, grid=8)`. Does position get even easier to read?

### 🗣️ Explain it back
"Why can't a bigger neural network fix state aliasing?" Answer in one sentence.

In [ ]:
progress_report()